In [ ]:
import json
from pathlib import Path

import pandas as pd

from uavloc.simulation.sequence import village_bootstrap
from uavloc.track import experiment_dir

# Params

In [ ]:
BOOTSTRAP_RESAMPLES = 10_000
SEED = 42
PAIRS = {"5 s": (("028-ins-error-state", "028.1"), ("027-sequence-filters", "027.2")),
         "10 s": (("030-fix-interval", "030.1"), ("030-fix-interval", "030.3")),
         "20 s": (("030-fix-interval", "030.2"), ("030-fix-interval", "030.4"))}
BAR_RATIO_VS_REPLACE = 0.8

# Data

In [ ]:
def flights(exp: str, run: str) -> pd.DataFrame:
    return pd.read_csv(experiment_dir(exp) / "artifacts" / f"flights-{run}.csv")

# Run

In [ ]:
rows = []
for interval, (filt, replace) in PAIRS.items():
    a, b = flights(*filt), flights(*replace)
    delta, low, high = village_bootstrap(a, b, "end_error_m", BOOTSTRAP_RESAMPLES, SEED)
    rows.append({"interval": interval, "filter_run": filt[1], "replace_run": replace[1],
                 "filter_mean_end_m": float(a["end_error_m"].mean()), "replace_mean_end_m": float(b["end_error_m"].mean()),
                 "ratio": float(a["end_error_m"].mean() / b["end_error_m"].mean()), "delta": delta, "ci_low": low, "ci_high": high})
lead = pd.DataFrame(rows)
lead.round(3)

In [ ]:
ten = lead.set_index("interval").loc["10 s"]
bar2 = {"ratio": float(ten["ratio"]), "ci_high": float(ten["ci_high"]),
        "pass": bool(ten["ratio"] <= BAR_RATIO_VS_REPLACE and ten["ci_high"] < 0)}
Path("artifacts/analysis.json").write_text(json.dumps({"lead": lead.to_dict("records"), "bar2": bar2}, indent=1))
bar2